# Shark Attack

In [2]:
import pandas as pd

import xlrd

df = pd.read_excel("GSAF5.xls")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df

## Phase 1 – Data Familiarization

Initial exploration of the dataset to understand its structure and identify potential data quality issues.

The dataset contains historical records of shark attacks, including information such as location, activity, age, gender, injury details, and date of the incident.

At first glance, the dataset appears to be unstructured and contains inconsistencies, missing values, and non-standardized text fields, which makes data cleaning a necessary step before analysis.

In [ ]:
df.head() # Initial exploration of the dataset

In [ ]:
df.info() # We observe missing values and inconsistent formats

Several data quality issues were identified:

Missing values in key columns such as Age, Activity, and Country
Inconsistent text formatting, especially in the Activity column (e.g., "surf", "surfing", "surfboard")
Unstructured text fields in columns like Injury and Species
Inconsistent date formats, making time-based analysis difficult
Potential duplicate records

These issues must be addressed before performing any reliable analysis.

## Phase 2 – Data Cleaning & Transformation



In [ ]:
#data cleaning column names step 1
df.columns=df.columns.str.strip()
print(df)



In [ ]:
#step 2 Handling null values

df.isnull().sum()

In [ ]:
# Drop obvious irrelevent columns
df = df.drop(columns=[
    "Unnamed: 21", "Unnamed: 22",
    "href", "href formula", "pdf",
    "Case Number.1"
], errors="ignore")
df

In [ ]:
#Remove columns where more than 50% of values are missing
df = df.dropna(axis=1, thresh=len(df) * 0.5)#keeps columns only with enough data
df


In [ ]:
# Remove any remaining unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df

In [ ]:
#Remove duplicate rows
df = df.drop_duplicates()
df

In [ ]:
#Handles missing values
df = df.fillna({
    "Country": "Unknown",
    "State": "Unknown",
    "Location": "Unknown",
    "Activity": "Unknown",
    "Sex": "Unknown",
    "Age": "Unknown",
    "Fatal Y/N": "Unknown",
    "Species": "Unknown"
})
df

In [ ]:
#manipulating strings
df["Sex"] = df["Sex"].str.upper().str.strip()
df["Fatal Y/N"] = df["Fatal Y/N"].str.upper().str.strip()
df

In [ ]:
#Regex to extract numerical ages
import re
df["Age_clean"] = (
    df["Age"]
    .astype(str)
    .apply(lambda x: re.findall(r"\d+", x))
    .apply(lambda x: int(x[0]) if x else None)
)
df

In [ ]:
#Extract Age 
df["Age_clean"] = df["Age"].str.extract(r"(\d+)")
df

In [ ]:
#Data cleaning in Date format
df["Date_clean"] = df["Date"].str.replace(
    r"(\d+)(st|nd|rd|th)", r"\1", regex=True
)
df

In [ ]:
#standartize Activity
df["Activity_group"] = df["Activity"].str.lower().apply(
    lambda x: "Swimming" if "swim" in x else
              "Surfing" if "surf" in x else
              "Fishing" if "fish" in x else
              "Diving" if "dive" in x else
              "Other"
)
df

In [ ]:
#standardize location
import re
df["Location"] = df["Location"].str.replace(r"Off|Near", "", regex=True, flags=re.IGNORECASE).str.strip()
df

In [ ]:
print(df["Location"].head(10))

## Phase 3 – Structuring & Exploratory Data Analysis (EDA)


In [19]:
# Remove 'st', 'nd', 'rd', 'th'
df["Date"] = df["Date"].astype(str).str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)

# Combine with Year column
df["Date_str"] = df["Date"] + " " + df["Year"].astype("Int64").astype(str)
df

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Fatal Y/N,Time,Species,Source,Case Number,original order,Age_clean,Date_clean,Activity_group,Date_str
0,18 March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,...,N,1715hrs,Great White Shark,US Sun: Mendocino Coast News:Melissa Michaelson:,NaN,NaN,39,18 March,Surfing,18 March 2026
1,14 March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,Unknown,F,?,...,N,1015hrs,Unknown shark 5ft (1.5m),People Magazine: Kevin McMurray Trackingsharks...,NaN,NaN,NaN,14 March,Swimming,14 March 2026
2,10 March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,Dave Daniell,M,?,...,N,0800hrs,Great White Shark 10ft (3m),Perth Now: Kevin McMurray Trackingsharks.com,NaN,NaN,NaN,10 March,Other,10 March 2026
3,5 March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,Unknown,M,50's,...,N,0815hrs,Unknown,News.com.au: ABC News: Andy Currie,NaN,NaN,50,5 March,Other,5 March 2026
4,22 February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata Point Magnin,Wing Foiling,Cyril Chevalier,M,55,...,Y,?,Tiger or bull shark implicated,Johannes Marchand: Andy Currie,NaN,NaN,55,22 February,Other,22 February 2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7077,Before 1903,0.0,Unprovoked,AUSTRALIA,Western Australia,Roebuck Bay,Diving,male,M,Unknown,...,Y,NaN,Unknown,"H. Taunton; N. Bartlett, p. 234",ND.0005,6.0,NaN,Before 1903,Other,Before 1903 0
7078,Before 1903,0.0,Unprovoked,AUSTRALIA,Western Australia,Unknown,Pearl diving,Ahmun,M,Unknown,...,Y,NaN,Unknown,"H. Taunton; N. Bartlett, pp. 233-234",ND.0004,5.0,NaN,Before 1903,Other,Before 1903 0
7079,1900-1905,0.0,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,Unknown,...,Y,NaN,Unknown,"F. Schwartz, p.23; C. Creswell, GSAF",ND.0003,4.0,NaN,1900-1905,Swimming,1900-1905 0
7080,1883-1889,0.0,Unprovoked,PANAMA,Unknown,"Panama Bay 8ºN, 79ºW",Unknown,Jules Patterson,M,Unknown,...,Y,NaN,Unknown,"The Sun, 10/20/1938",ND.0002,3.0,NaN,1883-1889,Other,1883-1889 0
